<a href="https://colab.research.google.com/github/Dharshanthisside/ITA0616---Machine-learning/blob/main/Exp_3_Decision_Tree_(ID3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# PlayTennis dataset (built in, no file needed)
data = pd.DataFrame({
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool',
                    'Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal',
                 'High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong',
             'Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes',
                   'No','Yes','Yes','Yes','Yes','Yes','No']
})

def entropy(col):
    p = col.value_counts(normalize=True)
    return -(p * np.log2(p)).sum()

def info_gain(df, attr, target):
    weighted = sum(len(sub) / len(df) * entropy(sub[target])
                   for _, sub in df.groupby(attr))
    return entropy(df[target]) - weighted

def id3(df, target, attrs):
    if df[target].nunique() == 1:
        return df[target].iloc[0]
    if not attrs:
        return df[target].mode()[0]
    best = max(attrs, key=lambda a: info_gain(df, a, target))
    tree = {best: {}}
    for value, subset in df.groupby(best):
        tree[best][value] = id3(subset, target, [a for a in attrs if a != best])
    return tree

def classify(tree, sample):
    while isinstance(tree, dict):
        attr = next(iter(tree))
        tree = tree[attr][sample[attr]]
    return tree

def print_tree(tree, indent=0):
    for attr, branches in tree.items():
        for value, sub in branches.items():
            print("  " * indent + f"{attr} = {value}", end="")
            if isinstance(sub, dict):
                print()
                print_tree(sub, indent + 1)
            else:
                print(f"  -> {sub}")

attrs = list(data.columns[:-1])
print("Information gain of each attribute:")
for a in attrs:
    print(f"  {a}: {info_gain(data, a, 'PlayTennis'):.4f}")

tree = id3(data, 'PlayTennis', attrs)
print("\nDecision Tree:")
print_tree(tree)

new_sample = {'Outlook': 'Sunny', 'Temperature': 'Cool',
              'Humidity': 'High', 'Wind': 'Strong'}
print("\nNew sample:", new_sample)
print("Predicted class:", classify(tree, new_sample))

Information gain of each attribute:
  Outlook: 0.2467
  Temperature: 0.0292
  Humidity: 0.1518
  Wind: 0.0481

Decision Tree:
Outlook = Overcast  -> Yes
Outlook = Rain
  Wind = Strong  -> No
  Wind = Weak  -> Yes
Outlook = Sunny
  Humidity = High  -> No
  Humidity = Normal  -> Yes

New sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted class: No
